# Laboratorio 08 — Neo4j I: pensar en grafos

**Procesamiento Masivo de Datos (ELE051-B / EIN102B)** · USM 2026-2 · Paralelo 701
**Clase 11** · Viernes 23 de octubre de 2026 · **Laboratorio para trabajar en casa**

---

En la Clase 11 (online) vimos estos ejercicios en vivo. Ahora les toca hacerlos a ustedes, a su ritmo: cuenten con 1,5–2 horas. No se entrega ni se evalúa, pero conviene hacerlo antes de la Clase 12, que parte de este mismo grafo.

Con Redis y HBase preguntábamos **por clave**: «dame la fila de la estación E-07». Hoy la pregunta es otra: **«¿con quién se conecta esto?»**. Con claves y valores, ese recorrido lo hace uno, a mano, consulta tras consulta. En Neo4j la **relación** es un dato de primera clase y el recorrido lo hace la base.

Pregunta guía: *¿a qué paradas llego desde Plaza Perú con dos tramos de micro?*

**Índice:** Parte 0 (levantar Neo4j) · Parte 1 (cargar el grafo) · Ejercicios 1–7 · Checkpoint · Síntesis.

## Cómo trabajar este notebook

- Solos o en equipos de 2–3 personas.
- **Las consultas son Cypher**, el lenguaje de Neo4j. Se escriben como texto dentro de `q("…")` y Python las envía a la base por el protocolo Bolt (puerto 7687) con el driver oficial `neo4j`. El resultado vuelve como tabla (`DataFrame`).
- **El Neo4j Browser** (<http://localhost:7474>, usuario `neo4j`, clave `pmd2026neo`) es la otra puerta a la misma base: ahí se pega el mismo Cypher y se ve **dibujado**. Conviene tenerlo abierto en otra pestaña.
- Cada ejercicio tiene dos momentos: **«Lo de la diapositiva»** (completo: ejecútenlo y miren) y **«Ahora ustedes»** (completar los `___`, guiándose por la pista).
- Referencia: [Cypher Manual](https://neo4j.com/docs/cypher-manual/current/) · [driver de Python](https://neo4j.com/docs/python-manual/current/).
- ⚠️ Descarguen la imagen **antes de abrir el notebook**: `docker pull neo4j:5-community` (~350 MB de descarga, ~1 GB en disco; verificar con `docker images | grep neo4j`).

## Dependencias

Instala lo de `requirements.txt` (el driver `neo4j`, `pandas` y `matplotlib`) en **el Python de este kernel**, y confirma que se importa. Si ya estaba todo, solo dice `Requirement already satisfied` o nada. Es `%pip`, no `!pip`: `%pip` instala en el ambiente del kernel, aunque no sea el de la terminal.

In [ ]:
%pip install -q -r requirements.txt
import matplotlib, neo4j, pandas
print("✓ neo4j", neo4j.__version__, "· pandas", pandas.__version__, "· matplotlib", matplotlib.__version__)

## Configuración

Ejecútenla una vez. `q()` envía una consulta y devuelve un `DataFrame`; `cargar()` ejecuta el script `01_cargar_grafo.cypher` sentencia por sentencia.

In [ ]:
import subprocess
from pathlib import Path

import pandas as pd
from neo4j import GraphDatabase
from neo4j.exceptions import ConstraintError

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "pmd2026neo")
driver = None   # se crea en la Parte 0


def q(cypher, **params):
    """Ejecuta una consulta Cypher y devuelve el resultado como DataFrame."""
    with driver.session() as s:
        return s.run(cypher, params).to_df()


def cargar(ruta="01_cargar_grafo.cypher"):
    """Ejecuta el script sentencia por sentencia (cada una en su propia transacción)."""
    texto = Path(ruta).read_text(encoding="utf-8")
    sentencias = []
    for trozo in texto.split(";"):
        lineas = [l for l in trozo.splitlines() if l.strip() and not l.strip().startswith("//")]
        if lineas:
            sentencias.append("\n".join(lineas))
    with driver.session() as s:
        for st in sentencias:
            s.run(st).consume()
    return len(sentencias)

---
# Parte 0 — Levantar Neo4j

`up -d --wait` levanta el contenedor (`docker-compose.yml` de esta carpeta) y espera a que Neo4j acepte consultas. Neo4j corre sobre Java: la primera vez tarda 30–60 s.

Expone dos puertos:

| Puerto | Qué |
|---|---|
| `7474` | Neo4j Browser (HTTP): <http://localhost:7474> |
| `7687` | Bolt: por donde hablan el driver de Python y `cypher-shell` |

In [ ]:
subprocess.run(["docker", "compose", "up", "-d", "--wait"], check=True)
driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("✓ Neo4j responde ·", q("CALL dbms.components() YIELD name, versions RETURN name, versions[0] AS version").to_dict("records"))

**Esperado:** `✓ Neo4j responde · [{'name': 'Neo4j Kernel', 'version': '5.26.…'}]`. Después abran <http://localhost:7474> en el navegador, conéctense con `bolt://localhost:7687`, usuario `neo4j` y clave `pmd2026neo`, y prueben `RETURN 1`.

**Si algo falla:**
- **Puerto ocupado:** `docker ps` y bajen el contenedor que sobra. No cambien el compose a ciegas.
- **`up --wait` espera ~2–3 min y termina con `container pmd-neo4j is unhealthy`** (en Python, `CalledProcessError`): casi siempre es un volumen viejo con otra clave. La clave de `NEO4J_AUTH` solo se aplica al crear el volumen. Solución: `docker compose down -v` y repetir.

---
# Parte 1 — Cargar el grafo y verificar

Una red **sintética** de paradas de micro del Gran Concepción (no es el GTFS real; las coordenadas son aproximadas). El modelo:

- **Nodos:** `(:Parada {id, nombre, lat, lon})`, `(:Comuna {nombre})`, `(:Linea {id, nombre})`.
- **Relaciones:** `(:Parada)-[:EN]->(:Comuna)`, `(:Linea)-[:PASA_POR {orden}]->(:Parada)` y `(:Parada)-[:SIGUE_A {linea, minutos}]->(:Parada)`.
- `SIGUE_A` apunta **en el sentido de la línea**. Para preguntar «conectado con» se recorre sin dirección: `-[:SIGUE_A]-`.

| id | nombre | comuna |
|---|---|---|
| P01 | Plaza Perú | Concepción |
| P02 | Universidad de Concepción | Concepción |
| P03 | Terminal Collao | Concepción |
| P04 | Barrio Norte | Concepción |
| P05 | Puente Llacolén | San Pedro de la Paz |
| P06 | Mall del Trébol | Talcahuano |
| P07 | Terminal Talcahuano | Talcahuano |
| P08 | Parque Pedro del Río | Hualpén |
| P09 | Plaza Chiguayante | Chiguayante |
| P10 | Plaza Penco | Penco |
| P11 | Plaza Coronel | Coronel |
| P12 | Lomas Coloradas | San Pedro de la Paz |

| Línea | Nombre | Paradas | Minutos por tramo |
|---|---|---|---|
| L1 | Concepción–Talcahuano | P01 → P06 → P08 → P07 | 12, 8, 10 |
| L2 | Concepción–Penco | P02 → P01 → P04 → P10 | 6, 7, 18 |
| L3 | Collao–Lomas Coloradas | P03 → P02 → P09 → P12 | 9, 15, 14 |
| L4 | Concepción–Coronel | P01 → P05 → P12 → P11 | 9, 11, 20 |

⚠️ El script se carga **una sola vez**. Si se repite, los `CREATE` chocan con las restricciones de unicidad (en el Ejercicio 5 vemos por qué). La celda revisa antes si ya está cargado.

### Lo de la diapositiva

In [ ]:
if q("MATCH (n) RETURN count(n) AS n")["n"][0] == 0:   # base vacía: cargar
    print(f"{cargar()} sentencias ejecutadas")
else:
    print("El grafo ya estaba cargado: no se repite (en el Ej. 5 vemos qué pasaría).")
q("MATCH (n) RETURN labels(n)[0] AS tipo, count(*) AS n ORDER BY tipo")

**Esperado:** `6 sentencias ejecutadas` y Comuna 7 · Linea 4 · Parada 12. Si da otro número: `q("MATCH (n) DETACH DELETE n")` y vuelvan a ejecutar la celda.

### Ahora ustedes

Cuenten las **relaciones** por tipo.

In [ ]:
q("MATCH ()-[r]->() RETURN ___ AS tipo, count(*) AS n ORDER BY tipo")   # Pista: type(r) devuelve el tipo de una relación

**Esperado:** EN 12 · PASA_POR 16 · SIGUE_A 12.

**✍️ Respuesta del equipo — 1:** ¿por qué hay 16 `PASA_POR` pero solo 12 `SIGUE_A`, si son las mismas 4 líneas? `___`

---
# Ejercicio 1 — Mira tu grafo

Este ejercicio se hace **en el Browser** (<http://localhost:7474>). Peguen esta consulta y activen la vista de **grafo** (no la de tabla):

```cypher
MATCH (p:Parada)-[r:SIGUE_A]->(q:Parada)
RETURN p, r, q;
```

Hagan clic en un nodo y en una flecha para ver sus propiedades. Después peguen `MATCH (n)-[r]->(m) RETURN n, r, m;` para ver el grafo completo, con comunas y líneas.

Lo mismo, preguntado desde Python:

In [ ]:
print("Etiquetas:", q("CALL db.labels()")["label"].tolist())
print("Tipos de relación:", q("CALL db.relationshipTypes()")["relationshipType"].tolist())

**Esperado:** 3 etiquetas (`Comuna`, `Linea`, `Parada`) y 3 tipos (`EN`, `PASA_POR`, `SIGUE_A`).

**✍️ Respuesta del equipo — 2:** ¿qué parada tiene más flechas? ¿Qué propiedades tiene una flecha `SIGUE_A`? `___`

---
# Ejercicio 2 — MATCH y WHERE

`MATCH` describe un **patrón**: `(nodo)-[relación]->(nodo)`, escrito casi como un dibujo. Las llaves `{…}` filtran por propiedades; un `-` sin flecha ignora la dirección.

### Lo de la diapositiva

(a) Las paradas de la comuna de Concepción:

In [ ]:
q("""
MATCH (p:Parada)-[:EN]->(c:Comuna {nombre: 'Concepción'})
RETURN p.nombre ORDER BY p.nombre
""")

(b) Los **vecinos directos** de Plaza Perú, con la línea y los minutos del tramo:

In [ ]:
q("""
MATCH (:Parada {id: 'P01'})-[r:SIGUE_A]-(v:Parada)
RETURN v.nombre AS vecino, r.linea AS linea, r.minutos AS minutos
ORDER BY minutos
""")

**Esperado:** (a) 4 filas: Barrio Norte, Plaza Perú, Terminal Collao y Universidad de Concepción. (b) 4 filas: Universidad de Concepción (L2, 6) · Barrio Norte (L2, 7) · Puente Llacolén (L4, 9) · Mall del Trébol (L1, 12).

### Ahora ustedes

In [ ]:
# (a') las paradas de San Pedro de la Paz
q("""
MATCH (p:Parada)-[:EN]->(c:Comuna {nombre: ___})
RETURN p.nombre AS parada ORDER BY parada
""")   # Pista: el nombre de la comuna, entre comillas simples y con mayúsculas exactas

In [ ]:
# (b') los vecinos de Lomas Coloradas, buscándola por NOMBRE (no por id)
q("""
MATCH (:Parada {nombre: 'Lomas Coloradas'})-[r:SIGUE_A]-(v:Parada)
RETURN v.nombre AS vecino, r.linea AS linea, r.minutos AS minutos
ORDER BY ___
""")   # Pista: igual que en la diapositiva, del tramo más corto al más largo

**Esperado:** (a') Lomas Coloradas y Puente Llacolén. (b') Puente Llacolén (L4, 11), Plaza Chiguayante (L3, 14) y Plaza Coronel (L4, 20).

**✍️ Respuesta del equipo — 3:** en (b') el patrón usa `-` sin flecha. ¿Qué vecinos se perderían con `->`? `___`

---
# Ejercicio 3 — Patrones de varios saltos

`*2` en la relación = caminos de **exactamente dos** tramos. `*1..3` = de uno a tres. En un camino no se repite una misma relación, pero sí se puede llegar a una parada por caminos distintos: `DISTINCT` evita que salga repetida.

### Lo de la diapositiva

**La pregunta guía, contestada:** ¿a qué paradas llego desde Plaza Perú con dos tramos?

In [ ]:
q("""
MATCH (:Parada {id: 'P01'})-[:SIGUE_A*2]-(q:Parada)
RETURN DISTINCT q.nombre AS parada ORDER BY parada
""")

**Esperado:** 5 filas: Lomas Coloradas, Parque Pedro del Río, Plaza Chiguayante, Plaza Penco y Terminal Collao.

### Ahora ustedes

In [ ]:
# (i) uno O dos tramos desde Plaza Perú: ¿cuántas paradas distintas?
q("""
MATCH (:Parada {id: 'P01'})-[:SIGUE_A ___]-(q:Parada)
RETURN count(DISTINCT q) AS paradas
""")   # Pista: el rango de largos se escribe *min..max

In [ ]:
# (ii) ¿hasta dónde llego desde Plaza Coronel con a lo más 4 tramos?
q("""
MATCH (:Parada {nombre: 'Plaza Coronel'})-[:SIGUE_A*1..4]-(q:Parada)
RETURN ___ q.nombre AS parada ORDER BY parada
""")   # Pista: sin esto, una parada alcanzada por dos caminos sale dos veces

**Esperado:** (i) 9. (ii) 8 paradas: Barrio Norte, Lomas Coloradas, Mall del Trébol, Plaza Chiguayante, Plaza Perú, Puente Llacolén, Terminal Collao y Universidad de Concepción. Pruébenlo **sin** lo que pusieron en el blanco: salen 10 filas, con Plaza Perú y Universidad de Concepción dos veces, porque a cada una se llega por los dos lados del ciclo P02–P09–P12–P05–P01.

**✍️ Respuesta del equipo — 4:** ¿cómo responderían (ii) en Redis o en HBase? `___`

---
# Ejercicio 4 — Transbordos

Un **transbordo** es una parada por la que pasa más de una línea. `collect` junta valores en una lista y `WITH … WHERE` filtra sobre el resultado ya agrupado.

### Lo de la diapositiva

In [ ]:
q("""
MATCH (l:Linea)-[:PASA_POR]->(p:Parada)
WITH p, collect(l.id) AS lineas
WHERE size(lineas) > 1
RETURN p.nombre AS parada, lineas
ORDER BY size(lineas) DESC, parada
""")

**Esperado:** 3 filas: Plaza Perú [L1, L2, L4] · Lomas Coloradas [L3, L4] · Universidad de Concepción [L2, L3]. El orden de las líneas dentro de cada lista puede variar.

### Ahora ustedes

In [ ]:
# ¿cuántas paradas tiene cada comuna?
q("""
MATCH (p:Parada)-[:EN]->(c:Comuna)
RETURN c.nombre AS comuna, ___ AS paradas
ORDER BY paradas DESC, comuna
""")   # Pista: la misma función que en la Parte 1. No hace falta GROUP BY: Cypher agrupa por lo que no es agregación

**Esperado:** Concepción 4 · San Pedro de la Paz 2 · Talcahuano 2 · Chiguayante 1 · Coronel 1 · Hualpén 1 · Penco 1.

*Buen punto para una pausa: si lo hacen en dos sesiones, pueden dejar el contenedor arriba (`docker compose stop` / `start` conserva el grafo).*

---
# Ejercicio 5 — Escribir datos: CREATE, MERGE, SET, DELETE

`CREATE` **siempre** crea. `MERGE` crea **solo si no existe** (si existe, lo encuentra). `SET` pone o cambia propiedades. `DETACH DELETE` borra un nodo junto con sus relaciones.

### Lo de la diapositiva

1. Crear la parada de prueba `P99`. La primera línea borra una P99 anterior, por si re-ejecutan la celda:

In [ ]:
q("MATCH (n:Parada {id: 'P99'}) DETACH DELETE n")   # por si re-ejecutan: parte limpio
q("CREATE (n:Parada {id: 'P99', nombre: 'Parada de prueba', lat: -36.82, lon: -73.04})")
q("MATCH (p:Parada) RETURN count(p) AS paradas")

**Esperado:** 13 paradas.

2. **El mismo `CREATE` otra vez.** ¿Qué pasa?

In [ ]:
try:
    q("CREATE (n:Parada {id: 'P99', nombre: 'Parada de prueba', lat: -36.82, lon: -73.04})")
except ConstraintError as e:
    print("✗ ConstraintError:", e.message.splitlines()[0])

3. Enlazar P99 a Concepción con `MERGE` (si la relación ya existe, no la duplica):

In [ ]:
q("""
MATCH (n:Parada {id: 'P99'}), (c:Comuna {nombre: 'Concepción'})
MERGE (n)-[:EN]->(c)
RETURN n.nombre AS parada, c.nombre AS comuna
""")

4. Ponerle una propiedad nueva. Ninguna otra parada tiene `techada`: en un grafo, cada nodo lleva las propiedades que necesita.

In [ ]:
q("MATCH (n:Parada {id: 'P99'}) SET n.techada = true RETURN properties(n) AS n")   # en el Browser: RETURN n, y se ve el nodo

5. Borrarla con sus relaciones y contar:

In [ ]:
q("MATCH (n:Parada {id: 'P99'}) DETACH DELETE n")
q("MATCH (p:Parada) RETURN count(p) AS paradas")

**Esperado:** 12.

### Ahora ustedes

P99 ya no existe. Ejecuten esta celda **dos veces** y miren el conteo cada vez.

In [ ]:
# MERGE en vez de CREATE: ejecútenla DOS veces y cuenten
q("___ (n:Parada {id: 'P99'})")   # Pista: el verbo que crea solo si no existe
q("MATCH (n:Parada {id: 'P99'}) RETURN count(n) AS cuantas_P99")

**Esperado:** `cuantas_P99 = 1` las dos veces, sin error. La 1.ª vez el `MERGE` la creó (venía de 0); la 2.ª la encontró y no duplicó.

Cierre: dejen el grafo como estaba.

In [ ]:
q("MATCH (n:Parada {id: 'P99'}) DETACH DELETE n")
q("MATCH (p:Parada) RETURN count(p) AS paradas")

**Esperado:** 12.

**✍️ Respuesta del equipo — 5:** ¿por qué el segundo `CREATE` falló? ¿Qué lo impidió exactamente? `___`

---
# Ejercicio 6 — El grado de cada parada

El **grado** de un nodo es cuántas conexiones tiene. Cuenten solo los tramos `SIGUE_A`, sin dirección.

### Lo de la diapositiva

In [ ]:
grados = q("""
MATCH (p:Parada)
RETURN p.nombre AS parada, COUNT { (p)-[:SIGUE_A]-() } AS conexiones
ORDER BY conexiones DESC, parada
""")
print("Suma de grados:", grados["conexiones"].sum())
grados.iloc[::-1].plot.barh(x="parada", y="conexiones", legend=False, title="Tramos SIGUE_A por parada")   # invertido: el hub queda arriba
grados

**Esperado:** el hub es Plaza Perú (4); luego Lomas Coloradas y Universidad de Concepción (3), cinco paradas con 2 y cuatro con 1. La suma de grados es 24 = 2 × 12 tramos: cada tramo suma 1 a cada uno de sus dos extremos.

### Ahora ustedes: ¿qué pasa si cerramos Plaza Perú?

Primero, cuántas paradas se alcanzan desde Terminal Talcahuano con Plaza Perú abierta:

In [ ]:
q("""
MATCH camino = (:Parada {id: 'P07'})-[:SIGUE_A*1..11]-(q:Parada)
RETURN count(DISTINCT q) AS alcanzables
""")   # con Plaza Perú abierta

Ahora, solo con los caminos que **no pasan** por Plaza Perú:

In [ ]:
q("""
MATCH camino = (:Parada {id: 'P07'})-[:SIGUE_A*1..11]-(q:Parada)
WHERE NONE(n IN nodes(camino) WHERE ___)
RETURN count(DISTINCT q) AS alcanzables, collect(DISTINCT q.nombre) AS cuales
""")   # Pista: ningún nodo n del camino puede ser Plaza Perú: escriban la condición «n es Plaza Perú» (su id es 'P01')

**Esperado:** 11 con Plaza Perú abierta y 2 sin ella (Mall del Trébol y Parque Pedro del Río).

**✍️ Respuesta del equipo — 6:** ¿por qué el patrón dice `*1..11` y no `*` a secas? `___`

---
# Ejercicio 7 (desafío) — ¿Hasta dónde llego?

Desde **Terminal Talcahuano**, ¿qué paradas se alcanzan con hasta 3 tramos? Primero sin dirección y después con dirección.

### Lo de la diapositiva

In [ ]:
# sin dirección
q("""
MATCH (:Parada {id: 'P07'})-[:SIGUE_A*1..3]-(q:Parada)
RETURN DISTINCT q.nombre ORDER BY q.nombre
""")

In [ ]:
# con dirección
q("""
MATCH (:Parada {id: 'P07'})-[:SIGUE_A*1..3]->(q:Parada)
RETURN DISTINCT q.nombre ORDER BY q.nombre
""")

**Esperado:** 3 filas sin dirección (Mall del Trébol, Parque Pedro del Río y Plaza Perú) y **0 filas** con `->`.

### Ahora ustedes

In [ ]:
# ¿quién LLEGA a Terminal Talcahuano en hasta 3 tramos, siguiendo el sentido de las líneas?
q("""
MATCH (:Parada {id: 'P07'})___[:SIGUE_A*1..3]-(q:Parada)
RETURN DISTINCT q.nombre AS parada ORDER BY parada
""")   # Pista: la flecha también se puede escribir apuntando hacia la izquierda

**Esperado:** Mall del Trébol, Parque Pedro del Río y Plaza Perú.

**✍️ Respuesta del equipo — 7:** ¿por qué `->` devolvió 0 filas y `<-` devolvió 3? ¿Qué dice eso sobre la decisión de modelar `SIGUE_A` en el sentido de la línea? `___`

---
# Checkpoint

Vuelve a ejecutar las consultas clave y las compara con lo esperado. Es la tabla de la diapositiva 13.

In [ ]:
chequeos = {
    "Parte 1 · nodos (Comuna, Linea, Parada)": (q("MATCH (n) RETURN labels(n)[0] AS t, count(*) AS n ORDER BY t")["n"].tolist(), [7, 4, 12]),
    "Ej. 1 · etiquetas / tipos de relación": ((len(q("CALL db.labels()")), len(q("CALL db.relationshipTypes()"))), (3, 3)),
    "Ej. 2 · paradas en Concepción": (len(q("MATCH (:Comuna {nombre:'Concepción'})<-[:EN]-(p) RETURN p")), 4),
    "Ej. 3 · a 2 tramos de Plaza Perú": (len(q("MATCH (:Parada {id:'P01'})-[:SIGUE_A*2]-(q) RETURN DISTINCT q")), 5),
    "Ej. 4 · transbordos": (len(q("MATCH (l:Linea)-[:PASA_POR]->(p) WITH p, count(l) AS k WHERE k > 1 RETURN p")), 3),
    "Ej. 5 · paradas tras limpiar P99": (q("MATCH (p:Parada) RETURN count(p) AS n")["n"][0], 12),
    "Ej. 6 · grado de Plaza Perú": (q("MATCH (p:Parada {id:'P01'}) RETURN COUNT { (p)-[:SIGUE_A]-() } AS g")["g"][0], 4),
    "Ej. 7 · desde Terminal Talcahuano con ->": (len(q("MATCH (:Parada {id:'P07'})-[:SIGUE_A*1..3]->(q) RETURN q")), 0),
}
for nombre, (obtenido, esperado) in chequeos.items():
    print("✓" if obtenido == esperado else "✗", nombre, "→", obtenido, "" if obtenido == esperado else f"(esperado {esperado})")

**Esperado:** 8 ✓. Si salen ✗ **a la vez** en «Parte 1» (13 paradas), «Ej. 2» (5 en Concepción) y «Ej. 5» (13), quedó P99 viva, enlazada a Concepción → `q("MATCH (n:Parada {id:'P99'}) DETACH DELETE n")` y vuelvan a ejecutar el Checkpoint.

---
# Síntesis del equipo

Tres líneas:

1. En su **proyecto semestral**, ¿qué cosas serían **nodos** y qué verbos serían **relaciones**? `___`
2. Una pregunta de su proyecto que sería **torpe en SQL y natural en Cypher**, y por qué. `___`
3. Algo que **no** pondrían en un grafo. `___`

---
### Limpieza (al terminar)

`down -v` apaga el contenedor y borra el volumen. No se pierde nada que importe: el Lab 09 vuelve a cargar todo en ~1 minuto con el mismo script. Si prefieren dejarlo, `docker compose down` (sin `-v`) conserva el grafo.

In [ ]:
driver.close()
subprocess.run(["docker", "compose", "down", "-v"], check=True)   # sin -v, el grafo queda en el volumen
print("Listo.")

---
*Fin del Laboratorio 08.*